# PlasticWatch — train YOLO11s on TACO (ML-2)

Runs on a Colab GPU (or a local CUDA box). Produces `best.pt` for `DETECTOR_MODE=real`,
per-class metrics, the F1-vs-confidence curve that sets the operating threshold, ten
failure images, and `ml/reports/metrics.md`.

**Honesty rules that apply here (CLAUDE.md §2):**

* Report the **real** numbers, per class, and the drop on the local street set. Never
  promise an accuracy figure — not in the deck, the README or the demo.
* The five classes are fixed (SPEC §6). Never add a person, vehicle or licence-plate class.
* The four plastic classes mean **likely plastic**, derived from `class_map.csv` — a mapping
  two people review, not a TACO label.
* The split is by TACO batch. Re-split randomly and every number below is inflated by
  near-duplicate leakage, and must not be quoted.

In [ ]:
# 1. Environment. Colab: Runtime -> Change runtime type -> GPU.
!pip -q install "ultralytics==8.3.*"
import torch, ultralytics
print("ultralytics", ultralytics.__version__, "| cuda:", torch.cuda.is_available(),
      torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU only")

## 2. Dataset

Bring the dataset the repo scripts produce — do **not** re-convert here, so the split stays
identical to the one the evaluation uses:

```bash
python ml/scripts/download_taco.py --out ml/data/taco          # resumable; Flickr is flaky
python ml/scripts/taco_to_yolo.py --taco ml/data/taco --out ml/data/yolo
```

Upload `ml/data/yolo` (images + labels + `data.yaml`) to the GPU machine, or mount Drive
and point `DATA` at it.

In [ ]:
from pathlib import Path
DATA = Path("/content/yolo/data.yaml")   # <- the data.yaml from taco_to_yolo.py
assert DATA.is_file(), f"{DATA} missing - copy ml/data/yolo here first"
print(DATA.read_text())

In [ ]:
# 3. Sanity check: class counts per split. Confirm the batch-based split survived the copy.
from collections import Counter
root = DATA.parent
for split in ("train", "val"):
    counts = Counter()
    for txt in (root / "labels" / split).glob("*.txt"):
        for line in txt.read_text().splitlines():
            counts[int(line.split()[0])] += 1
    n_images = len(list((root / "images" / split).iterdir()))
    print(split, dict(sorted(counts.items())), f"({n_images} images)")

In [ ]:
# 4. Train. YOLO11s, 640 px, default augmentations (SPEC §6).
from ultralytics import YOLO
model = YOLO("yolo11s.pt")
results = model.train(
    data=str(DATA), imgsz=640, epochs=90, batch=16, patience=20,
    project="runs", name="plasticwatch", seed=0, deterministic=True,
)
BEST = Path(results.save_dir) / "weights" / "best.pt"
print("best weights:", BEST)

In [ ]:
# 5. Per-class validation metrics - what may be quoted. Per class, never as one figure.
m = YOLO(str(BEST)).val(data=str(DATA), imgsz=640, conf=0.001, verbose=False)
names = m.names if isinstance(m.names, dict) else dict(enumerate(m.names))
rows = []
for i, c in enumerate(m.ap_class_index):
    p, r, ap50, ap = m.box.class_result(i)
    rows.append((names[int(c)], p, r, ap50, ap))
mp, mr, map50, map95 = m.box.mean_results()
rows.append(("all", mp, mr, map50, map95))
print(f"{'class':<20}{'P':>8}{'R':>8}{'mAP50':>8}{'mAP50-95':>10}")
for n, p, r, a50, a in rows:
    print(f"{n:<20}{p:>8.3f}{r:>8.3f}{a50:>8.3f}{a:>10.3f}")

In [ ]:
# 6. Operating threshold from the F1-vs-confidence curve (SPEC §6: ~0.25-0.40).
import matplotlib.pyplot as plt
conf_x, f1_y = None, None
for x, y, _xl, yl in m.box.curves_results:
    if "F1" in yl:
        conf_x, f1_y = x, (y.mean(0) if y.ndim > 1 else y)
best_i = int(f1_y.argmax())
THRESHOLD = round(float(conf_x[best_i]), 2)
print(f"peak mean F1 {f1_y[best_i]:.3f} at confidence {THRESHOLD}  ->  DETECTOR_CONF_THRESHOLD")
plt.figure(figsize=(6, 3.2))
plt.plot(conf_x, f1_y)
plt.axvline(THRESHOLD, ls="--")
plt.xlabel("confidence"); plt.ylabel("mean F1")
plt.title("F1 vs confidence (validation)"); plt.tight_layout(); plt.show()

In [ ]:
# 7. Ten failure images: the worst validation images by missed + spurious boxes.
# Keep them - they go in the deck, so the demo shows where the detector breaks.
import shutil, torch
from ultralytics.utils.metrics import box_iou

FAIL = Path("failures"); FAIL.mkdir(exist_ok=True)
det = YOLO(str(BEST))
scored = []
for img in sorted((DATA.parent / "images" / "val").iterdir()):
    label = DATA.parent / "labels" / "val" / f"{img.stem}.txt"
    truth = [l.split() for l in label.read_text().splitlines()] if label.is_file() else []
    pred = det.predict(str(img), imgsz=640, conf=THRESHOLD, verbose=False)[0]
    n_true, n_pred, matched = len(truth), len(pred.boxes), 0
    if n_true and n_pred:
        h, w = pred.orig_shape
        gt = torch.tensor([[(float(t[1]) - float(t[3]) / 2) * w, (float(t[2]) - float(t[4]) / 2) * h,
                            (float(t[1]) + float(t[3]) / 2) * w, (float(t[2]) + float(t[4]) / 2) * h]
                           for t in truth])
        matched = int((box_iou(gt, pred.boxes.xyxy.cpu()).max(1).values > 0.5).sum())
    scored.append(((n_true - matched) + (n_pred - matched), img, pred))

scored.sort(key=lambda s: -s[0])
for err, img, pred in scored[:10]:
    pred.save(filename=str(FAIL / f"err{err:02d}_{img.name}"))
print("worst validation images written to", FAIL.resolve())

In [ ]:
# 8. Export: best.pt for the backend, ONNX for the hosted backup.
shutil.copy2(BEST, "best.pt")
YOLO(str(BEST)).export(format="onnx", imgsz=640, opset=12)
print("download best.pt -> backend/weights/best.pt  (gitignored)")
print(f"then set: DETECTOR_MODE=real  DETECTOR_CONF_THRESHOLD={THRESHOLD}")

In [ ]:
# 9. metrics.md - real numbers only. Fill the local-set table with ml/scripts/eval.py
#    (--local) once the team's own photos are labelled; leave it empty rather than guessed.
lines = [
    "# Detector metrics (TACO, YOLO11s)", "",
    f"Weights: `best.pt` - imgsz 640 - operating confidence **{THRESHOLD}**",
    "Split: by TACO batch folder (never random) - see `ml/scripts/taco_to_yolo.py`.", "",
    "Measured on TACO's validation batches. This is not a promise about performance on our",
    "streets: the local-set table below is the number that matters for the demo. Four of the",
    "five classes are *likely plastic* via `class_map.csv`, not a TACO label.", "",
    "| class | P | R | mAP50 | mAP50-95 |", "|---|---|---|---|---|",
]
for n, p, r, a50, a in rows:
    lines.append(f"| {n} | {p:.3f} | {r:.3f} | {a50:.3f} | {a:.3f} |")
lines += ["", "## Local street set", "",
          "Run `python ml/scripts/eval.py --weights backend/weights/best.pt --data <taco data.yaml>"
          " --local <local data.yaml>` and paste both tables here, including the drop.", "",
          "## Failure images", "",
          "The ten worst validation images are in `ml/reports/failures/`.", ""]
Path("metrics.md").write_text("\n".join(lines), encoding="utf-8")
print("\n".join(lines[:12]))
print("... copy metrics.md and failures/ into ml/reports/")

## 10. Swap it into the backend (ML-3)

1. Copy `best.pt` to `backend/weights/best.pt` (gitignored — never commit weights).
2. Set `DETECTOR_MODE=real` and `DETECTOR_CONF_THRESHOLD` to the threshold above.
3. `make test` — the pipeline, dedupe, scoring and workflow tests must still pass: the
   detector contract (SPEC §6) is unchanged, only its source.
4. Re-run `python ml/scripts/eval.py …` to reproduce the table outside the notebook.

If CPU inference is slower than ~2 s/image, retrain with `yolo11n.pt` (SPEC §6) and repeat.
Do not ship a model you cannot measure.